# Data Preparation

What this script does:
1) Loads the 6 CSVs and standardizes participant_id.
2) Deduplicates participant-level tables (keeps the latest record per participant, where possible).
3) Builds questionnaire scale scores (NRQ, PIES, ECM, Paas) + Cronbach’s alpha reliabilities.
4) Builds mouse movement features from mouse_events (distance, speed, duration, sample count).
5) Builds hover features from mouse_hovers (total hover time, number of hovers, etc.).
6) Left-joins everything into one analysis-ready participant-level dataset.

Outputs (written to ./out):
- analysis_ready.csv
- mouse_features.csv
- hover_features.csv
- questionnaire_scores.csv
- reliability_report.json

In [1]:
from __future__ import annotations

import os
import re
import numpy as np
import pandas as pd

### Configuration

In [3]:

# Folder where the CSVs are stored
DATA_DIR = "/Users/marlenerueschoff/Documents/Uni/UzK Master/Masterarbeit/Experiment/masterthesis_experiment-1/Data"  # <- hier ggf. deinen Ordnerpfad eintragen

# File names for the 6 input tables
FILES = {
    "demographics": "demographics_0601.csv",
    "mouse_events": "mouse_events_0601.csv",
    "mouse_hovers": "mouse_hovers_0601.csv",
    "participants": "participants_0601.csv",
    "questionnaires": "questionnaires_0601.csv",
    "sessions": "sessions_0601.csv",
}

# Primary key used across all tables
ID_COL = "participant_id"

# Reverse-coded items
REVERSE_ITEMS = {
     "PIES": ["PIES2", "PIES3"],
     "ECM_CI":["CI3"]   
}

# Likert scale maximum used for reverse coding
LIKERT_MAX = {
    "PIES": 5,
    "NRQ": 7,
    "ECM": 7,
}

### Helpers

In [ ]:
# ----------------------------
# BASIC HELPERS
# ----------------------------

def read_csv(path: str) -> pd.DataFrame:
    """
    Read a CSV into a DataFrame and standardize participant_id (as string, stripped).
    """
    df = pd.read_csv(path)
    if ID_COL in df.columns:
        df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def to_dt(df: pd.DataFrame, col: str) -> None:
    """
    Convert a DataFrame column to datetime (UTC). Errors become NaT.
    This is useful for sorting / deduplicating "latest row per participant".
    """
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)


def dedupe_latest(df: pd.DataFrame, id_col: str, time_col: str | None) -> pd.DataFrame:
    """
    Deduplicate participant-level tables:

    - If a time_col exists: keep the LAST (most recent) row per participant_id.
    - If no time_col: keep the last occurrence per participant_id.

    This protects you from repeated submissions or table re-inserts.
    """
    if id_col not in df.columns:
        return df.copy()

    out = df.copy()

    if time_col and time_col in out.columns:
        to_dt(out, time_col)
        out = out.sort_values([id_col, time_col])
        out = out.drop_duplicates(subset=[id_col], keep="last")
    else:
        out = out.drop_duplicates(subset=[id_col], keep="last")

    return out


def cronbach_alpha(items_df: pd.DataFrame) -> float:
    """
    Compute Cronbach's alpha for a set of item columns.

    Requirements:
    - rows = participants
    - columns = items
    - numeric values

    Notes:
    - Drops rows with any missing values in the item set.
    - Returns NaN if not enough items or variance is zero.
    """
    x = items_df.apply(pd.to_numeric, errors="coerce")
    x = x.dropna(axis=0, how="any")

    k = x.shape[1]  # number of items
    if k < 2 or x.shape[0] < 2:
        return np.nan

    # variance for each item column
    item_vars = x.var(axis=0, ddof=1)
    # variance of the sum score (total)
    total_var = x.sum(axis=1).var(ddof=1)

    if total_var == 0:
        return np.nan

    alpha = (k / (k - 1)) * (1 - item_vars.sum() / total_var)
    return float(alpha)


def reverse_code(series: pd.Series, likert_max: int) -> pd.Series:
    """
    Reverse-code a Likert item.
    If likert_max=7: new = 8 - old.
    """
    s = pd.to_numeric(series, errors="coerce")
    return (likert_max + 1) - s


def safe_mean(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    """
    Row-wise mean across a list of columns, coercing to numeric.
    If cols is empty, returns a Series of NaNs.
    """
    if not cols:
        return pd.Series([np.nan] * len(df), index=df.index)

    x = df[cols].apply(pd.to_numeric, errors="coerce")
    return x.mean(axis=1)


def classify_mobile(user_agent: str) -> bool:
    """
    Quick heuristic to classify whether a user-agent suggests a mobile device.
    (You can refine this if your analysis wants strict device filtering.)
    """
    if not isinstance(user_agent, str):
        return False
    return bool(re.search(r"\bMobile\b|Android|iPhone|iPad", user_agent, re.IGNORECASE))



NameError: name 'df' is not defined

### Questionnaire Scoring

In [ ]:
def build_questionnaire_scores(q: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Create participant-level questionnaire scores + a reliability report.

    Input: questionnaires table (potentially multiple rows per participant).
    Output:
      - DataFrame: one row per participant with scale means
      - dict: Cronbach alpha values + item lists for transparency

    Expected columns (based on your file):
      - paas
      - NRQ_ECL1..NRQ_ECL3
      - NRQ_ICL1..NRQ_ICL4
      - PIES1..PIES11 (+ PIES_AC optional single item)
      - ECM_PU1..4, ECM_S1..4, ECM_CI1..3
      - updated_at (used to keep the latest submission)
    """
    q = q.copy()
    q[ID_COL] = q[ID_COL].astype(str).str.strip()
    to_dt(q, "updated_at")

    # Keep the latest questionnaire row per participant
    q = dedupe_latest(q, ID_COL, "updated_at")

    # Find item columns by regex patterns
    pies_cols     = [c for c in q.columns if re.fullmatch(r"PIES\d+", c)]
    nrq_ecl_cols  = [c for c in q.columns if re.fullmatch(r"NRQ_ECL\d+", c)]
    nrq_icl_cols  = [c for c in q.columns if re.fullmatch(r"NRQ_ICL\d+", c)]
    ecm_pu_cols   = [c for c in q.columns if re.fullmatch(r"ECM_PU\d+", c)]
    ecm_s_cols    = [c for c in q.columns if re.fullmatch(r"ECM_S\d+", c)]
    ecm_ci_cols   = [c for c in q.columns if re.fullmatch(r"ECM_CI\d+", c)]

    # Optional reverse-coding: apply only if you filled REVERSE_ITEMS
    if "PIES" in REVERSE_ITEMS and REVERSE_ITEMS["PIES"]:
        for c in REVERSE_ITEMS["PIES"]:
            if c in q.columns:
                q[c] = reverse_code(q[c], LIKERT_MAX.get("PIES", 7))

    # Output table: one row per participant with computed scores
    out = q[[ID_COL]].copy()

    # Single-item perceived difficulty (Paas)
    if "paas" in q.columns:
        out["paas"] = pd.to_numeric(q["paas"], errors="coerce")

    # NRQ means (Extraneous / Intrinsic cognitive load)
    out["NRQ_ECL_mean"] = safe_mean(q, nrq_ecl_cols)
    out["NRQ_ICL_mean"] = safe_mean(q, nrq_icl_cols)

    # PIES interest mean score
    out["PIES_mean"] = safe_mean(q, pies_cols)

    # Optional single item (as in your file): "PIES_AC"
    if "PIES_AC" in q.columns:
        out["PIES_AC"] = pd.to_numeric(q["PIES_AC"], errors="coerce")

    # ECM means by subscale
    out["ECM_PU_mean"] = safe_mean(q, ecm_pu_cols)
    out["ECM_S_mean"]  = safe_mean(q, ecm_s_cols)
    out["ECM_CI_mean"] = safe_mean(q, ecm_ci_cols)

    # Overall ECM mean across all items
    ecm_all = ecm_pu_cols + ecm_s_cols + ecm_ci_cols
    out["ECM_all_mean"] = safe_mean(q, ecm_all)

    # Reliability report (Cronbach's alpha)
    reliability = {
        "alpha_PIES":     cronbach_alpha(q[pies_cols]) if len(pies_cols) >= 2 else np.nan,
        "alpha_NRQ_ECL":  cronbach_alpha(q[nrq_ecl_cols]) if len(nrq_ecl_cols) >= 2 else np.nan,
        "alpha_NRQ_ICL":  cronbach_alpha(q[nrq_icl_cols]) if len(nrq_icl_cols) >= 2 else np.nan,
        "alpha_ECM_PU":   cronbach_alpha(q[ecm_pu_cols]) if len(ecm_pu_cols) >= 2 else np.nan,
        "alpha_ECM_S":    cronbach_alpha(q[ecm_s_cols]) if len(ecm_s_cols) >= 2 else np.nan,
        "alpha_ECM_CI":   cronbach_alpha(q[ecm_ci_cols]) if len(ecm_ci_cols) >= 2 else np.nan,
        "alpha_ECM_all":  cronbach_alpha(q[ecm_all]) if len(ecm_all) >= 2 else np.nan,
        "n_questionnaire_rows_after_dedupe": int(len(q)),
        "n_unique_participants": int(q[ID_COL].nunique()),
        "pies_items": pies_cols,
        "nrq_ecl_items": nrq_ecl_cols,
        "nrq_icl_items": nrq_icl_cols,
        "ecm_pu_items": ecm_pu_cols,
        "ecm_s_items": ecm_s_cols,
        "ecm_ci_items": ecm_ci_cols,
    }

    return out, reliability


